In [5]:
# SETUP: Import the environment variable loader
# The dotenv library lets us store API keys securely in a .env file instead of hardcoding them
# If you get an ImportError, make sure you've installed python-dotenv (pip install python-dotenv)

from dotenv import load_dotenv

In [8]:
# Load environment variables from the .env file
# override=True ensures we get the latest values even if we run this cell multiple times
# This should return True if the .env file was found and loaded successfully

load_dotenv(override=True)


True

In [11]:
# Verify that the OpenAI API key is properly loaded
# This is a crucial security check before making any API calls
# Note: If using other providers (Anthropic, Google, etc.), check their respective key names

import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"✓ Success! Your OpenAI API key is loaded")
    print(f"✓ Key starts with: {openai_api_key[:8]}...")
    print(f"✓ Key length: {len(openai_api_key)} characters")
else:
    print("✗ ERROR: OpenAI API key not found!")
    print("✗ Please check your .env file contains: OPENAI_API_KEY=your-key-here")
    print("✗ See SETUP_GUIDE.md for detailed instructions")


✓ Success! Your OpenAI API key is loaded
✓ Key starts with: sk-proj-...
✓ Key length: 164 characters


In [14]:
# Import the OpenAI client library
# This provides the interface to communicate with OpenAI's API
# Note: Many AI providers (Anthropic Claude, Google Gemini, etc.) have similar client libraries
# If you get an ImportError, run: pip install openai

from openai import OpenAI


In [15]:
# Create an OpenAI client instance
# This client will handle all our API requests
# The API key is automatically read from the OPENAI_API_KEY environment variable
# Think of this as creating your "connection" to OpenAI's servers

openai = OpenAI()


In [16]:
# ========================================
# STEP 1: Test Basic API Connection
# ========================================
# Create a simple message to verify everything is working correctly
# Messages are always a list of dictionaries with two keys:
#   - "role": who is speaking ("user", "assistant", or "system")
#   - "content": the actual message text

messages = [{"role": "user", "content": "What is 2+2?"}]


In [17]:
# Send our first request to the OpenAI API
# Model: gpt-4o-mini - fast, affordable, and highly capable for most tasks
# The response object contains the AI's reply plus metadata (tokens used, timing, etc.)
# We extract just the text content from response.choices[0].message.content

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

print("AI Response:")
print(response.choices[0].message.content)


AI Response:
2 + 2 equals 4.


In [19]:
# ========================================
# STEP 2: Build a Simple Agent Pattern
# ========================================
# What is an "agent"? It's when AI output becomes input for another AI call
# Think of it like a two-step process:
#   1. AI makes a decision (e.g., "what should I cook?")
#   2. AI executes based on that decision (e.g., "here's the recipe")
#
# This is powerful because the AI can autonomously break down complex tasks!

# First API Call: Ask AI to suggest a meal based on our constraints
# We're being very specific in our instructions to get a clean, usable output

prompt = """You're a nutritionist. Suggest ONE specific healthy dinner meal 
that takes less than 30 minutes to prepare and costs under $15 for 2 people. 
Respond with ONLY the meal name, like 'Lemon Garlic Shrimp Pasta' - nothing else."""

messages = [{"role": "user", "content": prompt}]


In [21]:
# Execute the first API call to get a meal suggestion
# The AI will analyze our constraints (time, budget, health) and choose an appropriate meal
# This response becomes the "bridge" to our next step - that's the agent pattern!

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

meal_suggestion = response.choices[0].message.content

print("=" * 50)
print("AI's Meal Suggestion:")
print("=" * 50)
print(meal_suggestion)
print("=" * 50)


AI's Meal Suggestion:
Quinoa Black Bean Salad with Avocado


In [22]:
# ========================================
# STEP 3: Use AI's Output as New Input (Agent Chain)
# ========================================
# This is THE KEY CONCEPT of agent patterns!
# We're taking the meal name the AI just created and using it to ask for more details
# The AI doesn't know we're "chaining" - it just sees a question about a meal
# But WE know the meal came from AI, making this an autonomous agent workflow

# Create a new message asking for a complete recipe for the AI-suggested meal
messages = [{"role": "user", "content": f"Provide a detailed recipe for {meal_suggestion}, including ingredients list and step-by-step cooking instructions."}]


In [23]:
# Execute the second API call to get the full recipe
# This completes the agent loop:
#   Input → AI Decision (meal choice) → AI Execution (recipe creation) → Output
# 
# This is exactly how complex AI agents work - they chain multiple AI calls together!

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

recipe = response.choices[0].message.content
print("\n" + "=" * 50)
print("Complete Recipe:")
print("=" * 50)
print(recipe)



Complete Recipe:
Here’s a delicious and healthy recipe for Quinoa Black Bean Salad with Avocado. This refreshing salad is packed with nutrients and can be served as a main or as a side dish. 

### Quinoa Black Bean Salad with Avocado

#### Ingredients:

**For the Salad:**
- 1 cup quinoa
- 2 cups water or vegetable broth
- 1 can (15 oz) black beans, drained and rinsed
- 1 cup corn (fresh, frozen, or canned)
- 1 medium red bell pepper, diced
- 1 medium cucumber, diced
- 1 medium red onion, finely diced
- 1 avocado, diced
- 1 cup cherry tomatoes, halved
- 1/4 cup fresh cilantro, chopped (optional)
- Salt and pepper to taste

**For the Dressing:**
- 1/4 cup olive oil
- 2 tablespoons lime juice (about 1 lime)
- 1 teaspoon cumin
- 1 teaspoon chili powder (optional)
- 1 garlic clove, minced
- Salt to taste

#### Instructions:

1. **Cook the Quinoa:**
   - Rinse the quinoa under cold water in a fine-mesh strainer to remove any bitterness.
   - In a medium saucepan, combine the rinsed quinoa a

In [ ]:
# Display the recipe with beautiful Markdown formatting
# Markdown rendering makes text more readable with headers, lists, bold text, etc.
# Much better than plain text for structured content like recipes!

from IPython.display import Markdown, display

print("\n" + "=" * 50)
print("📖 FORMATTED RECIPE")
print("=" * 50 + "\n")

display(Markdown(recipe))

# ========================================
# 🎉 CONGRATULATIONS! You've Built an Agent!
# ========================================
#
# What just happened:
# 1. AI made an autonomous decision (which meal to suggest)
# 2. AI executed a task based on that decision (creating the recipe)
# 3. You didn't hardcode the meal name - the AI chose it dynamically!
#
# This is the foundation of modern AI agents. Scale this pattern up and you get:
#
# 🔬 Research Agents:
#    - Find relevant topics → Write detailed summaries → Generate insights
#
# 💻 Code Generation Agents:
#    - Plan software architecture → Write code modules → Generate tests
#
# ✍️ Content Creation Agents:
#    - Outline article structure → Write each section → Edit and polish
#
# 🧩 Problem-Solving Agents:
#    - Break down complex problems → Solve each part → Synthesize solution
#
# 🤖 Autonomous Agents (like AutoGPT):
#    - Chain dozens of AI calls together to complete complex multi-step tasks
#
# Next steps: Try modifying the prompts to create different agent behaviors!
